# 03. 이미지 전처리 (증강)

01 에서 눈으로 확인한 사실을 **가설로 바꾸고, 실험으로 판정한다.**

### 이 데이터의 특수 사정

수업 5장에서 X-ray 에 회전 증강을 주지 않은 이유는 "촬영 자세가 정해져 있어서"였다.
**같은 논리를 여기 적용하면 결론이 반대가 된다.**

| 증강 | 판단 | 이유 |
|---|---|---|
| 좌우/상하 반전 | **쓴다** | 세포는 방향성이 없다. 뒤집어도 같은 세포다 (X-ray 의 심장 위치와 다르다) |
| 회전(추가) | **검증 대상** | 데이터에 이미 회전이 걸려 있다 (01-3). 또 주면 패딩 위에 패딩이 쌓인다 |
| 중앙 자르기 | **검증 대상** | 회전 패딩을 없앨 수 있지만, 가장자리 세포를 잘라낼 위험도 있다 (01-4) |
| 색 변화 | **약하게만** | 염색 색조가 클래스 신호다. 호산구의 분홍 과립이 대표적 |
| RandomErasing | **불리할 것** | 화면에 판단 근거가 세포 하나뿐인데 그걸 가리면 라벨과 무관한 이미지가 된다 |

**증강은 많이 줄수록 좋은 게 아니라, 데이터가 실제로 가질 수 있는 변형만 주는 것이다.**

| 절 | 내용 |
|---|---|
| 3-1 | 변환을 직접 만들어 본다 (코드) |
| 3-2 | 프리셋 7종을 눈으로 비교 |
| 3-3 | 프리셋 7종을 같은 조건에서 학습해 비교 |
| 3-4 | 시드 5개 반복 — 08 가설검정 2의 재료 |

In [ ]:
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
from torchvision import transforms
from PIL import Image

import wbc
wbc.use_korean_font()

REF_MODEL   = 'resnet18'    # 전처리를 비교하는 동안 모델은 고정한다
IMAGE_SIZE  = 224
BATCH_SIZE  = 64
SCREEN_EPOCHS = 6           # 스크리닝은 짧게 — 지금 필요한 건 순위다
SUBSET      = None          # 더 빨리 돌리려면 4000 등으로 (순위 경향은 유지된다)
wbc.NUM_WORKERS = 4

print('기준 모델:', REF_MODEL, '| 장치:', wbc.device)

## 3-1. 변환을 직접 만들어 본다

`wbc.build_transforms()` 안에서 실제로 만들어지는 것이 아래와 같다.
**증강이 있는 것과 없는 것, 두 벌**을 만드는 것이 원칙이다 (수업 3·5장과 동일).
검증·테스트에는 **절대 증강을 걸지 않는다.**

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# 학습용 — 증강 있음
train_tf = transforms.Compose([
    transforms.CenterCrop((192, 256)),                    # 240x320 의 중앙 80% = 회전 패딩 제거
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),                    # 세포는 방향성이 없다
    transforms.RandomVerticalFlip(),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.01),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),    # ImageNet 통계 — 사전학습 백본을 쓰므로
])

# 평가용 — 증강 없음
eval_tf = transforms.Compose([
    transforms.CenterCrop((192, 256)),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

d = os.path.join(wbc.DATA_DIR, 'TRAIN', 'NEUTROPHIL')
raw = Image.open(os.path.join(d, sorted(os.listdir(d))[0])).convert('RGB')

fig, ax = plt.subplots(1, 6, figsize=(15, 2.8))
ax[0].imshow(raw); ax[0].set_title('원본'); ax[0].axis('off')
ax[1].imshow(wbc._to_numpy_img(eval_tf(raw))); ax[1].set_title('평가용(증강 없음)'); ax[1].axis('off')
torch.manual_seed(0)
for k in range(2, 6):
    ax[k].imshow(wbc._to_numpy_img(train_tf(raw))); ax[k].set_title(f'학습용 {k-1}'); ax[k].axis('off')
plt.tight_layout(); plt.show()

**정규화 값이 ImageNet 통계인 이유**: 사전학습 백본이 그 분포를 기준으로 학습됐기 때문이다.
밑바닥 CNN 만 쓴다면 이 데이터 자체의 평균·표준편차를 써도 된다.

**`Normalize` 는 항상 `ToTensor` 뒤에** 온다. 순서가 바뀌면 PIL 이미지에 산술을 하려다 오류가 난다.

## 3-2. 프리셋 7종 — 눈으로 비교

`wbc.PRESETS` 에 정의된 7가지다. 같은 이미지에 각각을 5번씩 적용해 본다.
**같은 프리셋인데 매번 결과가 달라야 정상이다** (무작위 증강이므로).

In [ ]:
for name, p in wbc.PRESETS.items():
    print(f'{name:9s} 반전 {str(p["flip"]):5s} 회전 {p["affine"]:2d}° 색 {p["color"]:.1f} '
          f'자르기 {p["crop"]:.2f} 지우기 {p["erase"]:.2f}')

In [ ]:
names = list(wbc.PRESETS)
fig, axes = plt.subplots(len(names), 5, figsize=(12, 2.3 * len(names)))
for r, p in enumerate(names):
    tf, _ = wbc.build_transforms(p, 160)
    torch.manual_seed(0)
    for k in range(5):
        axes[r, k].imshow(wbc._to_numpy_img(tf(raw))); axes[r, k].axis('off')
    axes[r, 0].set_title(p, loc='left', fontsize=11)
plt.tight_layout(); plt.show()

## 3-3. 프리셋 7종 학습 비교

**조건을 하나만 바꾼다.** 모델·학습률·에폭·시드를 전부 고정하고 전처리만 바꾼다.
그렇지 않으면 무엇이 효과를 냈는지 말할 수 없다.

그리고 **모든 판단은 검증셋으로만** 한다. 테스트셋은 06 노트북에서 딱 한 번 연다.

In [ ]:
for p in wbc.PRESETS:
    wbc.run_experiment(f'P_{p}', model_name=REF_MODEL, preset=p, image_size=IMAGE_SIZE,
                       batch_size=BATCH_SIZE, lr=3e-4, epochs=SCREEN_EPOCHS,
                       patience=3, seed=42, subset=SUBSET)
    print('-' * 78)

In [ ]:
t = wbc.runs_table()
pre = t[t.run_id.str.startswith('P_')].sort_values('val_macro_f1', ascending=False)
display(pre[['run_id', 'preset', 'best_epoch', 'epoch_sec',
             'val_accuracy', 'val_macro_f1', 'val_auc']].round(4))

base = float(pre[pre.preset == 'none'].val_macro_f1.iloc[0])
plt.figure(figsize=(7.5, 3.6))
diff = pre.val_macro_f1 - base
plt.bar(pre.preset, diff, color=['tab:green' if v > 0 else 'tab:red' for v in diff])
plt.axhline(0, c='k', lw=.8); plt.ylabel('증강 없음 대비 macro-F1 차이')
plt.title('전처리 프리셋 효과 (기준: none)'); plt.xticks(rotation=20); plt.grid(alpha=.3, axis='y')
plt.show()

BEST_PRESET = pre.iloc[0].preset
wbc.save_cfg(preset=BEST_PRESET)          # 뒤 노트북들이 config.json 에서 읽어 간다

### 해석할 때 주의할 점

여기서 나온 1등을 곧바로 "통계적으로 더 좋다"고 말하면 안 된다.
**시드 하나로 한 번씩 돌린 결과**라 0.002 정도의 차이는 우연일 수 있다.

- 지금 단계 = **후보 좁히기** (순위)
- 3-4 에서 **시드 5개로 반복**해 점수를 모으고
- **08 가설검정 2** 에서 대응표본 t검정 / 윌콕슨 검정으로 판정한다

가설이 맞았는지도 함께 확인한다.
- `geo`(강한 회전)가 `flip` 보다 나쁘다면 → **"이미 회전 증강된 데이터"** 가설이 지지된 것
- `color`(강한 색 변화)가 나쁘다면 → **"색이 클래스 신호"** 가설이 지지된 것
- `crop` 이 좋다면 → **"검은 패딩이 방해였다"** 가설이 지지된 것

**예상이 빗나갔다면 그것도 결과다.** 왜 빗나갔는지 쓰는 게 더 좋은 보고서다.

## 3-4. 시드 5개 반복 — 가설검정용 자료 수집

08 의 가설검정 2 로 이어지는 다리다.

같은 설정이라도 초기화·데이터 순서가 달라지면 점수가 흔들린다.
그 흔들림보다 전처리의 효과가 큰지 보려면 **여러 번 돌린 분포**가 필요하다.

- **조건 A**: 3-3 에서 고른 최적 전처리
- **조건 B**: 증강 없음(`none`)

같은 시드끼리 짝지어 5쌍을 만든다.

> **H0**: 두 전처리의 검증 macro-F1 평균이 같다
> **H1**: 다르다 (양측검정)

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
REPEAT_EPOCHS = 8
AB = {'A_' + BEST_PRESET: BEST_PRESET, 'B_none': 'none'}
print(f'총 {len(SEEDS)*len(AB)*REPEAT_EPOCHS} 에폭 예정. 에폭당 60초면 약 '
      f'{len(SEEDS)*len(AB)*REPEAT_EPOCHS/60:.0f}분.')

for tag, preset in AB.items():
    for s in SEEDS:
        wbc.run_experiment(f'SEED_{tag}_s{s}', model_name=REF_MODEL, preset=preset,
                           image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, lr=3e-4,
                           epochs=REPEAT_EPOCHS, patience=3, seed=s, subset=SUBSET, verbose=False)
        print(f'  {tag} seed={s} 완료')

In [ ]:
t = wbc.runs_table()
sd = t[t.run_id.str.startswith('SEED_')].copy()
sd['조건'] = np.where(sd.preset == 'none', 'B_증강없음', 'A_최적전처리')
piv = sd.pivot_table(index='seed', columns='조건', values='val_macro_f1')
display(piv.round(4)); display(piv.describe().round(4).loc[['mean', 'std']])

plt.figure(figsize=(5.5, 3.8))
for _, r in piv.iterrows():
    plt.plot(['A_최적전처리', 'B_증강없음'], [r['A_최적전처리'], r['B_증강없음']],
             marker='o', color='gray', alpha=.7)
plt.boxplot([piv['A_최적전처리'], piv['B_증강없음']], positions=[0, 1], widths=.35)
plt.ylabel('검증 macro-F1'); plt.title('시드 5개 반복 (같은 시드끼리 선으로 연결)')
plt.grid(alpha=.3); plt.show()

piv.to_csv('seed_scores.csv', encoding='utf-8-sig')
print('seed_scores.csv 저장 — 08 노트북이 이 값으로 대응표본 검정을 한다')

## 03 정리

- 전처리 프리셋 7종을 **같은 조건**에서 비교했고, 선택 근거를 데이터의 성질로 설명할 수 있다
- 선택한 프리셋을 `config.json` 에 저장했다 (04~06 이 읽어 간다)
- 시드 5개 반복 점수를 `seed_scores.csv` 에 저장했다 (08 가설검정 2)

→ 다음: **04_모델비교_선정.ipynb**